In [1]:
# Optimizing thresholds on the training data

import numpy as np
import pandas as pd

from dotenv import load_dotenv
import dagshub
import mlflow
import mlflow.lightgbm

from src.config import DAGSHUB_USERNAME, DAGSHUB_REPO
from src.config import TARGET, TIME_COL, UID_COL, processed_dataset
from src.evaluation.optimize_fraud_thresholds import optimize_fraud_thresholds



load_dotenv()

dagshub.init(
    repo_owner=DAGSHUB_USERNAME,
    repo_name=DAGSHUB_REPO,
    mlflow=True
    )

mlflow.set_experiment("ieee-fraud-inference")
print(f"Connected to MLflow: {mlflow.get_tracking_uri()}")

# Load model and feature list
model_uri="models:/m-095e643b2f31472682dca4eacebe030c"
print(f"Fetching model from: {model_uri}")
model = mlflow.lightgbm.load_model(model_uri)

df = pd.read_parquet(processed_dataset)
df = df.sort_values(TIME_COL).reset_index(drop=True)

# 80/20 chronological holdout (matches NN model)
n_samples = len(df)
split_idx = int(n_samples * 0.8)
idx_val_time = np.arange(split_idx, n_samples)

y_val = df[TARGET].iloc[idx_val_time].to_numpy(dtype=np.float32)
print(f"Validation set size: {len(y_val):,} rows | Fraud rate: {y_val.mean():.4f}")

te_features = ['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M4']
drop_cols = [
    TARGET, 
    TIME_COL, 
    UID_COL
] + te_features  # Drops raw ProductCD, card4, etc.

features = [c for c in df.columns if c not in drop_cols]

td_cols = df[features].select_dtypes(include=["timedelta", "datetime"]).columns.tolist()
for c in td_cols:
    df[c] = df[c].dt.total_seconds().astype("float32")

X_va_lgb = df.iloc[idx_val_time][features]
val_amounts=df.iloc[idx_val_time]['TransactionAmt'].to_numpy(dtype=np.float32)

lgb_val_preds = model.predict(X_va_lgb)

optimal_policy=optimize_fraud_thresholds(
    y_prob=lgb_val_preds,
    amounts=val_amounts,
    y_true=y_val,
    save_path="artifacts/thresholds.json"
)

print("\n--- OPTIMAL POLICY RESULTS ---")
for k, v in optimal_policy.items():
    print(f"{k}: {v:,.4f}" if isinstance(v, float) else f"{k}: {v}")

c:\Users\ADMIN\Documents\DataScience\ieee-fraud-detection\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Accessing as anamish05

Initialized MLflow to track repo "anamish05/ieee-fraud-detection"

Repository anamish05/ieee-fraud-detection initialized!

Connected to MLflow: https://dagshub.com/anamish05/ieee-fraud-detection.mlflow
Fetching model from: models:/m-095e643b2f31472682dca4eacebe030c


Validation set size: 118,108 rows | Fraud rate: 0.0344
Optimal policy saved to artifacts\thresholds.json

--- OPTIMAL POLICY RESULTS ---
t_approve: 0.0386
t_decline: 0.2431


In [3]:
from src.streaming.streamer import TransactionStreamer
from src.config import processed_dataset, raw_data_path, artifacts_path


streamer = TransactionStreamer(
        model_uri="models:/m-095e643b2f31472682dca4eacebe030c",
        features_path="artifacts/model_features.json",
        target_enc_path="artifacts/target_encodings.json",
        freq_enc_path="artifacts/freq_encodings.json",
        cat_features_path="artifacts/cat_features.json",
        train_warmup_path=processed_dataset,
        thresholds_path=artifacts_path / 'thresholds.json'
    )



Hydrating 24h training table...


Initialized MLflow to track repo "anamish05/ieee-fraud-detection"

Repository anamish05/ieee-fraud-detection initialized!

Connected to MLflow: https://dagshub.com/anamish05/ieee-fraud-detection.mlflow
Fetching model from: models:/m-095e643b2f31472682dca4eacebe030c


In [4]:
streamer.run_inference(
        test_trans_path=raw_data_path / "test_transaction.csv",
        test_ident_path=raw_data_path / "test_identity.csv",
        output_csv="reports/test_submission.csv"
    )

Inference complete: output saved to reports/test_submission.csv
Inference complete: output saved to reports/test_submission.csv

=== BUSINESS DECISION SUMMARY ===
total_transactions: 506691
auto_approve_rate: 0.8457
manual_review_rate: 0.1210
hard_decline_rate: 0.0333
total_euros_declined: 3,009,826.2780
total_euros_routed_to_review: 16,186,181.5950
manual_review_operational_cost: 153,295.0000
prevented_fraud_loss: 1,576,806.8754
expected_fraud_loss: 806,291.4872
churn_friction_loss: 69,492.9842
total_portfolio_cost: 1,029,079.4714
policy_roi_per_review_euro: 10.2861
Uploading submission to DagsHub MLflow Artifacts...
🏃 View run streaming-inference at: https://dagshub.com/anamish05/ieee-fraud-detection.mlflow/#/experiments/3/runs/a3c932a3e30c4cf79638891390297b29
🧪 View experiment at: https://dagshub.com/anamish05/ieee-fraud-detection.mlflow/#/experiments/3
